In [0]:
from datetime import date, timedelta
from pyspark.sql import SparkSession
from pyspark.sql.types import *

spark = SparkSession.builder.getOrCreate()

# ── Helpers ───────────────────────────────────────────────────────────────────

def get_easter(y):
    a,b,c = y%19, y//100, y%100
    d,e   = b//4, b%4
    f,g   = (b+8)//25, (b-(b+8)//25+1)//3
    h     = (19*a+b-d-g+15)%30
    i,k   = c//4, c%4
    l     = (32+2*e+2*i-h-k)%7
    m     = (a+11*h+22*l)//451
    mo    = (h+l-7*m+114)//31
    dy    = ((h+l-7*m+114)%31)+1
    return date(y, mo, dy)

def nth_mon(y, mo, n):
    d = date(y, mo, 1); cnt = 0
    while d.month == mo:
        if d.weekday() == 0:
            cnt += 1
            if cnt == n: return d
        d += timedelta(1)

def last_mon_on_or_before(y, mo, dy):
    d = date(y, mo, dy)
    while d.weekday() != 0:
        d -= timedelta(1)
    return d

def observed(d):
    if d.weekday() == 6: return d + timedelta(1)
    if d.weekday() == 5: return d + timedelta(2)
    return d

def build_holidays(y):
    e   = get_easter(y)
    nyd = observed(date(y, 1,  1))
    fd  = nth_mon(y, 2, 3)
    gf  = e - timedelta(2)
    em  = e + timedelta(1)
    vd  = last_mon_on_or_before(y, 5, 25)
    cd  = observed(date(y, 7,  1))
    ch  = nth_mon(y, 8, 1)
    ld  = nth_mon(y, 9, 1)
    trd = date(y, 9, 30)
    tk  = nth_mon(y, 10, 2)
    rd  = date(y, 11, 11)
    xd  = date(y, 12, 25)
    bd  = date(y, 12, 26)
    return [
        (nyd, "New Year's Day",             "Federal",        True ),
        (fd,  "Family Day",                 "ON/AB/BC/SK",    True ),
        (gf,  "Good Friday",                "Federal",        True ),
        (em,  "Easter Monday",              "Most provinces", False),
        (vd,  "Victoria Day",               "Federal",        True ),
        (cd,  "Canada Day",                 "Federal",        cd.weekday() in [0, 4]),
        (ch,  "Civic/Heritage Day",         "AB/BC/ON/NS",    True ),
        (ld,  "Labour Day",                 "Federal",        True ),
        (trd, "Truth & Reconciliation Day", "Federal govt+",  False),
        (tk,  "Thanksgiving",               "Federal",        True ),
        (rd,  "Remembrance Day",            "Federal govt+",  False),
        (xd,  "Christmas Day",              "Federal",        True ),
        (bd,  "Boxing Day",                 "Federal",        True ),
    ]

hol_lookup = {}
for y in range(2022, 2027):
    for dt, name, scope, lw in build_holidays(y):
        hol_lookup[dt] = (name, scope, lw)

def season(m):
    if m in [12, 1, 2]: return "winter"
    if m == 3:           return "transition"
    if m in [4, 5]:      return "spring"
    if m in [6, 7, 8]:   return "summer"
    return "fall"

# ── Build rows ────────────────────────────────────────────────────────────────

rows = []
current = date(2022, 12, 26)
end     = date(2026, 12, 28)

while current <= end:
    ws = current
    we = current + timedelta(6)
    m  = ws.month
    y  = ws.year

    hols    = []
    lw_hols = []
    for i in range(7):
        dd = ws + timedelta(i)
        if dd in hol_lookup:
            name, scope, lw = hol_lookup[dd]
            hols.append((dd, name, scope, lw))
            if lw:
                lw_hols.append(name)

    iso_yr, iso_wk, _ = ws.isocalendar()

    xmas = date(y, 12, 25)
    days_to_xmas = (xmas - ws).days
    if days_to_xmas < 0:
        days_to_xmas = (date(y + 1, 12, 25) - ws).days

    rows.append((
        iso_yr,
        f"W{iso_wk:02d}",
        ws.strftime("%Y-%m-%d"),
        we.strftime("%Y-%m-%d"),
        m,
        ws.strftime("%B"),
        f"Q{(m - 1) // 3 + 1}",
        season(m),
        days_to_xmas,
        "Y" if hols else "N",
        " | ".join(h[1] for h in hols),
        " | ".join(h[0].strftime("%b %d") for h in hols),
        " | ".join(h[2] for h in hols),
        "Y" if lw_hols else "N",
        " | ".join(lw_hols),
    ))

    current += timedelta(7)

# ── Create Spark DataFrame ────────────────────────────────────────────────────

schema = StructType([
    StructField("iso_year",             IntegerType(), False),
    StructField("iso_week",             StringType(),  False),
    StructField("week_start_mon",       StringType(),  False),
    StructField("week_end_sun",         StringType(),  False),
    StructField("month_num",            IntegerType(), False),
    StructField("month_name",           StringType(),  False),
    StructField("quarter",              StringType(),  False),
    StructField("season",               StringType(),  False),
    StructField("days_to_christmas",    IntegerType(), False),
    StructField("stat_holiday_in_week", StringType(),  False),
    StructField("stat_holiday_names",   StringType(),  False),
    StructField("stat_holiday_dates",   StringType(),  False),
    StructField("stat_holiday_scopes",  StringType(),  False),
    StructField("is_long_weekend",      StringType(),  False),
    StructField("long_weekend_driver",  StringType(),  False),
])

df = spark.createDataFrame(rows, schema)

df.display()

In [0]:
import requests
from collections import defaultdict
from datetime import date, timedelta
from pyspark.sql import SparkSession
from pyspark.sql.types import *

spark = SparkSession.builder.getOrCreate()

# ── Pull holidays from canada-holidays.ca ─────────────────────────────────────
# API docs: https://canada-holidays.ca/api
# Response shape: {"holidays": [{"date","nameEn","federal","observedDate","provinces":[{"id",...}]}]}

API = "https://canada-holidays.ca/api/v1/holidays"

# Business rule: these holidays exist but Cineplex shouldn't treat them as
# long-weekend drivers (not widely observed as days off in retail/entertainment).
NON_LW_DRIVERS = {
    "Easter Monday",
    "National Day for Truth and Reconciliation",
    "Remembrance Day",
}

def fetch_holidays(year):
    r = requests.get(API, params={"year": year}, timeout=30)
    r.raise_for_status()
    return r.json()["holidays"]

# date -> list of (name, scope, is_long_weekend_driver)
hol_lookup = defaultdict(list)

for y in range(2022, 2027):
    for h in fetch_holidays(y):
        obs   = date.fromisoformat(h["observedDate"])
        name  = h["nameEn"]
        provs = sorted({p["id"] for p in h["provinces"]})
        # Federal flag wins; otherwise show the provincial list
        scope = "Federal" if h.get("federal") == 1 else "/".join(provs)
        # Long weekend = observed Mon or Fri, excluding non-driver holidays
        is_lw = obs.weekday() in (0, 4) and name not in NON_LW_DRIVERS
        hol_lookup[obs].append((name, scope, is_lw))

def season(m):
    if m in (12, 1, 2): return "winter"
    if m == 3:          return "transition"
    if m in (4, 5):     return "spring"
    if m in (6, 7, 8):  return "summer"
    return "fall"

# ── Build weekly rows ─────────────────────────────────────────────────────────

rows    = []
current = date(2022, 12, 26)
end     = date(2026, 12, 28)

while current <= end:
    ws, we = current, current + timedelta(6)
    m, y   = ws.month, ws.year

    hols, lw_hols = [], []
    for i in range(7):
        dd = ws + timedelta(i)
        for name, scope, lw in hol_lookup.get(dd, []):
            hols.append((dd, name, scope, lw))
            if lw:
                lw_hols.append(name)

    iso_yr, iso_wk, _ = ws.isocalendar()

    xmas = date(y, 12, 25)
    days_to_xmas = (xmas - ws).days
    if days_to_xmas < 0:
        days_to_xmas = (date(y + 1, 12, 25) - ws).days

    rows.append((
        iso_yr,
        f"W{iso_wk:02d}",
        ws.strftime("%Y-%m-%d"),
        we.strftime("%Y-%m-%d"),
        m,
        ws.strftime("%B"),
        f"Q{(m - 1) // 3 + 1}",
        season(m),
        days_to_xmas,
        "Y" if hols else "N",
        " | ".join(h[1] for h in hols),
        " | ".join(h[0].strftime("%b %d") for h in hols),
        " | ".join(h[2] for h in hols),
        "Y" if lw_hols else "N",
        " | ".join(lw_hols),
    ))

    current += timedelta(7)

# ── Schema + DataFrame ────────────────────────────────────────────────────────

schema = StructType([
    StructField("iso_year",             IntegerType(), False),
    StructField("iso_week",             StringType(),  False),
    StructField("week_start_mon",       StringType(),  False),
    StructField("week_end_sun",         StringType(),  False),
    StructField("month_num",            IntegerType(), False),
    StructField("month_name",           StringType(),  False),
    StructField("quarter",              StringType(),  False),
    StructField("season",               StringType(),  False),
    StructField("days_to_christmas",    IntegerType(), False),
    StructField("stat_holiday_in_week", StringType(),  False),
    StructField("stat_holiday_names",   StringType(),  False),
    StructField("stat_holiday_dates",   StringType(),  False),
    StructField("stat_holiday_scopes",  StringType(),  False),
    StructField("is_long_weekend",      StringType(),  False),
    StructField("long_weekend_driver",  StringType(),  False),
])

df = spark.createDataFrame(rows, schema)
df.display()

In [0]:
# ============================================================
# OPEN-METEO WEATHER PIPELINE
# FINAL PRODUCTION MMM VERSION
# ============================================================

# ============================================================
# IMPORTS
# ============================================================

import pandas as pd
import numpy as np
import requests
import time

from concurrent.futures import ThreadPoolExecutor


# ============================================================
# CONFIG
# ============================================================

START_DATE = "2022-12-26"

END_DATE = (
    pd.Timestamp.today()
    .strftime("%Y-%m-%d")
)

BASE_URL = (
    "https://archive-api.open-meteo.com/v1/archive"
)

MAX_WORKERS = 2


# ============================================================
# CITY CONFIG
# ============================================================

cities = {

    "Toronto": {
        "latitude": 43.6532,
        "longitude": -79.3832,
        "timezone": "America/Toronto",
        "weight": 0.38
    },

    "Montreal": {
        "latitude": 45.5019,
        "longitude": -73.5674,
        "timezone": "America/Toronto",
        "weight": 0.18
    },

    "Vancouver": {
        "latitude": 49.2827,
        "longitude": -123.1207,
        "timezone": "America/Vancouver",
        "weight": 0.14
    },

    "Calgary": {
        "latitude": 51.0447,
        "longitude": -114.0719,
        "timezone": "America/Edmonton",
        "weight": 0.09
    },

    "Edmonton": {
        "latitude": 53.5461,
        "longitude": -113.4938,
        "timezone": "America/Edmonton",
        "weight": 0.07
    },

    "Ottawa": {
        "latitude": 45.4215,
        "longitude": -75.6972,
        "timezone": "America/Toronto",
        "weight": 0.05
    },

    "Winnipeg": {
        "latitude": 49.8951,
        "longitude": -97.1384,
        "timezone": "America/Winnipeg",
        "weight": 0.04
    },

    "Halifax": {
        "latitude": 44.6488,
        "longitude": -63.5752,
        "timezone": "America/Halifax",
        "weight": 0.03
    },

    "QuebecCity": {
        "latitude": 46.8139,
        "longitude": -71.2080,
        "timezone": "America/Toronto",
        "weight": 0.02
    }
}


# ============================================================
# FETCH WEATHER DATA
# ============================================================

def fetch_weather(city, config):

    print(f"Pulling {city}")

    params = {

        "latitude":
            config["latitude"],

        "longitude":
            config["longitude"],

        "start_date":
            START_DATE,

        "end_date":
            END_DATE,

        "timezone":
            config["timezone"],

        "daily": [

            # -------------------------
            # TEMPERATURE
            # -------------------------

            "temperature_2m_mean",
            "temperature_2m_max",
            "temperature_2m_min",

            # -------------------------
            # PRECIPITATION
            # -------------------------

            "precipitation_sum",
            "rain_sum",
            "snowfall_sum",

            # -------------------------
            # SUNLIGHT
            # -------------------------

            "daylight_duration",
            "sunshine_duration"
        ]
    }

    MAX_RETRIES = 5

    for attempt in range(MAX_RETRIES):

        try:

            response = requests.get(
                BASE_URL,
                params=params,
                timeout=120
            )

            if response.status_code == 200:

                data = response.json()

                if "daily" not in data:

                    print(f"NO DATA: {city}")

                    return pd.DataFrame()

                df = pd.DataFrame(
                    data["daily"]
                )

                if df.empty:

                    print(f"EMPTY DF: {city}")

                    return pd.DataFrame()

                df["city"] = city

                df["weight"] = config["weight"]

                print(f"SUCCESS: {city}")

                return df

            else:

                print(
                    f"Attempt {attempt+1} FAILED: "
                    f"{city} | Status {response.status_code}"
                )

        except Exception as e:

            print(
                f"Attempt {attempt+1} ERROR: "
                f"{city} | {e}"
            )

        sleep_time = 2 ** attempt

        time.sleep(sleep_time)

    print(f"FINAL FAILURE: {city}")

    return pd.DataFrame()


# ============================================================
# API PULL
# ============================================================

all_dfs = []

with ThreadPoolExecutor(
    max_workers=MAX_WORKERS
) as executor:

    futures = {

        executor.submit(
            fetch_weather,
            city,
            config
        ): city

        for city, config in cities.items()
    }

    for future in futures:

        result = future.result()

        if not result.empty:

            all_dfs.append(result)


# ============================================================
# CONCATENATE
# ============================================================

weather_df = pd.concat(
    all_dfs,
    ignore_index=True
)

print("Rows Pulled:", len(weather_df))


# ============================================================
# DATE FEATURES
# ============================================================

weather_df["date"] = pd.to_datetime(
    weather_df["time"]
)

weather_df["year"] = (
    weather_df["date"]
    .dt.year
)

weather_df["month"] = (
    weather_df["date"]
    .dt.month
)

weather_df["week"] = (
    weather_df["date"]
    .dt.isocalendar()
    .week
)

weather_df["day_of_week"] = (
    weather_df["date"]
    .dt.dayofweek
)


# ============================================================
# UNIT CONVERSION
# ============================================================

weather_df["daylight_hours"] = (
    weather_df["daylight_duration"] / 3600
)

weather_df["sunshine_hours"] = (
    weather_df["sunshine_duration"] / 3600
)


# ============================================================
# CLIMATOLOGY BASELINE
# ============================================================

monthly_climatology = (

    weather_df

    .groupby("month")["temperature_2m_mean"]

    .mean()

    .reset_index()

    .rename(columns={
        "temperature_2m_mean":
            "climatology_mean_temp"
    })
)

weather_df = weather_df.merge(
    monthly_climatology,
    on="month",
    how="left"
)


# ============================================================
# TEMPERATURE ANOMALY
# ============================================================

weather_df["temp_anomaly"] = (

    weather_df["temperature_2m_mean"]

    -

    weather_df["climatology_mean_temp"]
)


# ============================================================
# HDD / CDD
# ============================================================

BASE_TEMP = 18

weather_df["hdd"] = np.maximum(
    BASE_TEMP - weather_df["temperature_2m_mean"],
    0
)

weather_df["cdd"] = np.maximum(
    weather_df["temperature_2m_mean"] - BASE_TEMP,
    0
)


# ============================================================
# WEIGHTED FEATURES
# ============================================================

weighted_cols = [

    "temperature_2m_mean",
    "temperature_2m_max",
    "temperature_2m_min",

    "climatology_mean_temp",
    "temp_anomaly",

    "precipitation_sum",
    "rain_sum",
    "snowfall_sum",

    "daylight_hours",
    "sunshine_hours",

    "hdd",
    "cdd"
]

for col in weighted_cols:

    weather_df[f"weighted_{col}"] = (

        weather_df[col]

        *

        weather_df["weight"]
    )


# ============================================================
# NATIONAL DAILY AGGREGATION
# ============================================================

national_daily_weather = (

    weather_df

    .groupby("date")

    .agg({

        "weighted_temperature_2m_mean":
            "sum",

        "weighted_temperature_2m_max":
            "sum",

        "weighted_temperature_2m_min":
            "sum",

        "weighted_climatology_mean_temp":
            "sum",

        "weighted_temp_anomaly":
            "sum",

        "weighted_precipitation_sum":
            "sum",

        "weighted_rain_sum":
            "sum",

        "weighted_snowfall_sum":
            "sum",

        "weighted_daylight_hours":
            "sum",

        "weighted_sunshine_hours":
            "sum",

        "weighted_hdd":
            "sum",

        "weighted_cdd":
            "sum"
    })

    .reset_index()
)


# ============================================================
# RENAME DAILY FEATURES
# ============================================================

national_daily_weather = national_daily_weather.rename(columns={

    "weighted_temperature_2m_mean":
        "national_mean_temp",

    "weighted_temperature_2m_max":
        "national_max_temp",

    "weighted_temperature_2m_min":
        "national_min_temp",

    "weighted_climatology_mean_temp":
        "national_climatology_temp",

    "weighted_temp_anomaly":
        "national_temp_anomaly",

    "weighted_precipitation_sum":
        "national_precip_mm",

    "weighted_rain_sum":
        "national_rain_mm",

    "weighted_snowfall_sum":
        "national_snow_cm",

    "weighted_daylight_hours":
        "national_daylight_hours",

    "weighted_sunshine_hours":
        "national_sunshine_hours",

    "weighted_hdd":
        "national_hdd",

    "weighted_cdd":
        "national_cdd"
})


# ============================================================
# SORT DAILY
# ============================================================

national_daily_weather = (

    national_daily_weather

    .sort_values("date")

    .reset_index(drop=True)
)


# ============================================================
# WEEK START (MONDAY)
# ============================================================

national_daily_weather["week_start"] = (

    national_daily_weather["date"]

    -

    pd.to_timedelta(
        national_daily_weather["date"].dt.weekday,
        unit="D"
    )
)


# ============================================================
# WEEKLY AGGREGATION
# ============================================================

weekly_weather = (

    national_daily_weather

    .groupby("week_start")

    .agg({

        "national_mean_temp":
            "mean",

        "national_max_temp":
            "max",

        "national_min_temp":
            "min",

        "national_climatology_temp":
            "mean",

        "national_temp_anomaly":
            "mean",

        "national_precip_mm":
            "sum",

        "national_rain_mm":
            "sum",

        "national_snow_cm":
            "sum",

        "national_daylight_hours":
            "mean",

        "national_sunshine_hours":
            "mean",

        "national_hdd":
            "sum",

        "national_cdd":
            "sum"
    })

    .reset_index()
)


# ============================================================
# TRUE WEEKLY ROLLING FEATURES
# ============================================================

weekly_weather["rain_2wk"] = (

    weekly_weather["national_rain_mm"]

    .rolling(
        window=2,
        min_periods=1
    )

    .sum()
)

weekly_weather["rain_4wk"] = (

    weekly_weather["national_rain_mm"]

    .rolling(
        window=4,
        min_periods=1
    )

    .sum()
)

weekly_weather["snow_2wk"] = (

    weekly_weather["national_snow_cm"]

    .rolling(
        window=2,
        min_periods=1
    )

    .sum()
)

weekly_weather["snow_4wk"] = (

    weekly_weather["national_snow_cm"]

    .rolling(
        window=4,
        min_periods=1
    )

    .sum()
)


# ============================================================
# LAG FEATURES
# ============================================================

weekly_weather["temp_anomaly_lag1"] = (
    weekly_weather["national_temp_anomaly"]
    .shift(1)
)

weekly_weather["rain_lag1"] = (
    weekly_weather["national_rain_mm"]
    .shift(1)
)

weekly_weather["snow_lag1"] = (
    weekly_weather["national_snow_cm"]
    .shift(1)
)


# ============================================================
# DYNAMIC THRESHOLDS
# ============================================================

RAIN_THRESHOLD = (
    weekly_weather["national_rain_mm"]
    .quantile(0.85)
)

SNOW_THRESHOLD = (
    weekly_weather["national_snow_cm"]
    .quantile(0.85)
)

HEAT_THRESHOLD = (
    weekly_weather["national_max_temp"]
    .quantile(0.90)
)

COLD_THRESHOLD = (
    weekly_weather["national_min_temp"]
    .quantile(0.10)
)


# ============================================================
# WEATHER FLAGS
# ============================================================

weekly_weather["heavy_rain_flag"] = np.where(

    weekly_weather["national_rain_mm"]

    >=

    RAIN_THRESHOLD,

    1,

    0
)

weekly_weather["heavy_snow_flag"] = np.where(

    weekly_weather["national_snow_cm"]

    >=

    SNOW_THRESHOLD,

    1,

    0
)

weekly_weather["extreme_heat_flag"] = np.where(

    weekly_weather["national_max_temp"]

    >=

    HEAT_THRESHOLD,

    1,

    0
)

weekly_weather["extreme_cold_flag"] = np.where(

    weekly_weather["national_min_temp"]

    <=

    COLD_THRESHOLD,

    1,

    0
)

weekly_weather["bad_weather_flag"] = np.where(

    (
        weekly_weather["heavy_rain_flag"] == 1
    )

    |

    (
        weekly_weather["heavy_snow_flag"] == 1
    )

    |

    (
        weekly_weather["extreme_cold_flag"] == 1
    ),

    1,

    0
)


# ============================================================
# Z-SCORES
# ============================================================

zscore_cols = [

    "national_temp_anomaly",
    "national_rain_mm",
    "national_snow_cm",
    "national_precip_mm"
]

for col in zscore_cols:

    weekly_weather[f"{col}_zscore"] = (

        weekly_weather[col]

        -

        weekly_weather[col].mean()

    ) / weekly_weather[col].std()


# ============================================================
# YEAR / WEEK FEATURES
# ============================================================

iso_calendar = (
    weekly_weather["week_start"]
    .dt.isocalendar()
)

weekly_weather["year"] = (
    iso_calendar.year
)

weekly_weather["week"] = (
    iso_calendar.week
)


# ============================================================
# ROUND ALL NUMERIC COLUMNS
# ============================================================

numeric_cols = weekly_weather.select_dtypes(
    include=[np.number]
).columns

weekly_weather[numeric_cols] = (
    weekly_weather[numeric_cols]
    .round(2)
)


# ============================================================
# FINAL SORT
# ============================================================

weekly_weather = (

    weekly_weather

    .sort_values("week_start")

    .reset_index(drop=True)
)


# ============================================================
# CONVERT week_start TO DATE ONLY
# ============================================================

weekly_weather["week_start"] = (
    pd.to_datetime(
        weekly_weather["week_start"]
    ).dt.date
)


# ============================================================
# PREVIEW
# ============================================================

print("\nWeekly Weather Preview")

print(

    weekly_weather[

        [

            "week_start",

            "national_mean_temp",
            "national_temp_anomaly",

            "national_rain_mm",
            "national_snow_cm",

            "heavy_rain_flag",
            "heavy_snow_flag",

            "extreme_heat_flag",
            "extreme_cold_flag",

            "bad_weather_flag"
        ]

    ].head(20)
)

print("\nDONE")


# ============================================================
# COLUMN DESCRIPTIONS
# ============================================================

column_descriptions = {

    "week_start":
        "Monday start date of the week",

    "national_mean_temp":
        "Weighted national average weekly temperature (C)",

    "national_max_temp":
        "Highest weighted national temperature during the week (C)",

    "national_min_temp":
        "Lowest weighted national temperature during the week (C)",

    "national_climatology_temp":
        "Historical monthly baseline temperature (C)",

    "national_temp_anomaly":
        "Difference between actual and climatology temperature (C)",

    "national_precip_mm":
        "Total weighted national precipitation (mm)",

    "national_rain_mm":
        "Total weighted national rainfall (mm)",

    "national_snow_cm":
        "Total weighted national snowfall (cm)",

    "national_daylight_hours":
        "Average weighted daylight hours",

    "national_sunshine_hours":
        "Average weighted sunshine hours",

    "national_hdd":
        "Heating Degree Days",

    "national_cdd":
        "Cooling Degree Days",

    "rain_2wk":
        "Rolling 2-week rainfall total (mm)",

    "rain_4wk":
        "Rolling 4-week rainfall total (mm)",

    "snow_2wk":
        "Rolling 2-week snowfall total (cm)",

    "snow_4wk":
        "Rolling 4-week snowfall total (cm)",

    "temp_anomaly_lag1":
        "Previous week's temperature anomaly",

    "rain_lag1":
        "Previous week's rainfall total",

    "snow_lag1":
        "Previous week's snowfall total",

    "heavy_rain_flag":
        "1 if rainfall exceeded 85th percentile threshold",

    "heavy_snow_flag":
        "1 if snowfall exceeded 85th percentile threshold",

    "extreme_heat_flag":
        "1 if max temperature exceeded 90th percentile threshold",

    "extreme_cold_flag":
        "1 if min temperature below 10th percentile threshold",

    "bad_weather_flag":
        "Combined severe weather indicator",

    "national_temp_anomaly_zscore":
        "Z-score standardized temperature anomaly",

    "national_rain_mm_zscore":
        "Z-score standardized rainfall",

    "national_snow_cm_zscore":
        "Z-score standardized snowfall",

    "national_precip_mm_zscore":
        "Z-score standardized precipitation",

    "year":
        "ISO calendar year",

    "week":
        "ISO calendar week number"
}

print("\nCOLUMN DESCRIPTIONS\n")

for col, desc in column_descriptions.items():

    print(f"{col}: {desc}")

In [0]:
weekly_weather["national_sunshine_ratio"] = (

    weekly_weather["national_sunshine_hours"]

    /

    weekly_weather["national_daylight_hours"]

).round(2)

# ============================================================
# REMOVE REDUNDANT FEATURES
# ============================================================

# ------------------------------------------------------------
# FEATURES TO DROP
# ------------------------------------------------------------
#
# Reasoning:
#
# 1. Highly correlated / redundant
#    - precip already includes rain + snow
#    - climatology temp used only to derive anomaly
#
# 2. Weak MMM contribution
#    - sunshine often noisy
#    - HDD/CDD redundant with temperature
#
# 3. Derived duplication
#    - lag + rolling together can overinflate weather impact
#
# ------------------------------------------------------------

drop_cols = [

    # -------------------------
    # REDUNDANT WEATHER
    # -------------------------

    "national_precip_mm",

    "national_climatology_temp",

    # -------------------------
    # REDUNDANT SUN FEATURES
    # -------------------------

    # "national_sunshine_hours",

    # -------------------------
    # REDUNDANT TEMP FEATURES
    # -------------------------

    "national_hdd",
    "national_cdd",

    # -------------------------
    # REDUNDANT ROLLING
    # -------------------------

    "rain_2wk",
    "snow_2wk",

    # -------------------------
    # REDUNDANT LAGS
    # -------------------------

    "rain_lag1",
    "snow_lag1",

    # -------------------------
    # REDUNDANT ZSCORES
    # -------------------------

    "national_precip_mm_zscore"
]


# ============================================================
# DROP EXISTING COLUMNS ONLY
# ============================================================

drop_cols_existing = [

    col for col in drop_cols

    if col in weekly_weather.columns
]

weekly_weather = weekly_weather.drop(
    columns=drop_cols_existing
)


# ============================================================
# FINAL COLUMN ORDER
# ============================================================

final_column_order = [

    # -------------------------
    # TIME
    # -------------------------

    "week_start",
    # "year",
    # "week",

    # -------------------------
    # CORE WEATHER
    # -------------------------

    "national_mean_temp",
    "national_max_temp",
    "national_min_temp",

    "national_temp_anomaly",
    # "temp_anomaly_lag1",

    "national_rain_mm",
    "national_snow_cm",

    # "rain_4wk",
    # "snow_4wk",

    # -------------------------
    # SUNLIGHT
    # -------------------------

    "national_sunshine_ratio",

    # -------------------------
    # FLAGS
    # -------------------------

    # "heavy_rain_flag",
    # "heavy_snow_flag",

    # "extreme_heat_flag",
    # "extreme_cold_flag",

    "bad_weather_flag",

    # -------------------------
    # STANDARDIZED FEATURES
    # -------------------------

    # "national_temp_anomaly_zscore",
    # "national_rain_mm_zscore",
    # "national_snow_cm_zscore"
]


# ============================================================
# KEEP EXISTING COLUMNS ONLY
# ============================================================

final_column_order = [

    col for col in final_column_order

    if col in weekly_weather.columns
]

weekly_weather = weekly_weather[
    final_column_order
]


In [0]:
# ============================================================
# MERGE ALL SEASONALITY FEATURES
# ============================================================

# Convert holidays Spark DF to pandas
holidays_pdf = df.toPandas()
holidays_pdf["week_start"] = pd.to_datetime(holidays_pdf["week_start_mon"])

# Ensure join key types match
weekly_weather["week_start"] = pd.to_datetime(weekly_weather["week_start"])
weekly_cpi["week_start"] = pd.to_datetime(weekly_cpi["week_start"])

# Merge holidays + weather
final_seasonality = holidays_pdf.merge(
    weekly_weather,
    on="week_start",
    how="left"
)

# Merge CPI
final_seasonality = final_seasonality.merge(
    weekly_cpi,
    on="week_start",
    how="left"
)

# Drop redundant column
final_seasonality = final_seasonality.drop(columns=["week_start_mon"])

# Sort
final_seasonality = final_seasonality.sort_values("week_start").reset_index(drop=True)

# Cut off at end of July 2025
final_seasonality = final_seasonality[final_seasonality["week_start"] <= "2025-07-28"].reset_index(drop=True)

print(f"Shape: {final_seasonality.shape}")
print(f"Columns: {list(final_seasonality.columns)}")
final_seasonality.display()

In [0]:
weekly_weather.display()

In [0]:
# ============================================================
# BANK OF CANADA CPI API
# FINAL WORKING VERSION
# ============================================================

import pandas as pd
import requests


# ============================================================
# BANK OF CANADA VALET API
# ============================================================

url = (
    "https://www.bankofcanada.ca/valet/"
    "observations/group/CPI_MONTHLY/json"
)


# ============================================================
# FETCH JSON
# ============================================================

response = requests.get(url)

json_data = response.json()


# ============================================================
# EXTRACT OBSERVATIONS
# ============================================================

cpi_df = pd.DataFrame(
    json_data["observations"]
)


# ============================================================
# KEEP REQUIRED COLUMNS
# ============================================================

cpi_df = cpi_df[
    [
        "d",
        "STATIC_TOTALCPICHANGE",
        "CPI_TRIM"
    ]
]


# ============================================================
# RENAME COLUMNS
# ============================================================

cpi_df = cpi_df.rename(columns={

    "d":
        "month_start",

    "STATIC_TOTALCPICHANGE":
        "headline_cpi",

    "CPI_TRIM":
        "core_cpi_trim"
})


# ============================================================
# EXTRACT VALUES
# ============================================================

for col in [

    "headline_cpi",
    "core_cpi_trim"
]:

    cpi_df[col] = cpi_df[col].apply(

        lambda x:

        x["v"]

        if isinstance(x, dict)

        else None
    )


# ============================================================
# DATE FORMAT
# ============================================================

cpi_df["month_start"] = pd.to_datetime(
    cpi_df["month_start"]
)


# ============================================================
# NUMERIC FORMAT
# ============================================================

numeric_cols = [

    "headline_cpi",
    "core_cpi_trim"
]

for col in numeric_cols:

    cpi_df[col] = pd.to_numeric(

        cpi_df[col],

        errors="coerce"
    ).round(2)


# ============================================================
# FILTER DATE RANGE
# ============================================================

cpi_df = cpi_df[
    cpi_df["month_start"] >= "2022-12-01"
]


# ============================================================
# CREATE WEEKLY DATE RANGE
# ============================================================

weekly_cpi = pd.DataFrame({

    "week_start": pd.date_range(

        start="2022-12-26",

        end=pd.Timestamp.today(),

        freq="W-MON"
    )
})


# ============================================================
# MONTH KEY
# ============================================================

weekly_cpi["month_key"] = (

    weekly_cpi["week_start"]

    .dt.strftime("%Y-%m")
)

cpi_df["month_key"] = (

    cpi_df["month_start"]

    .dt.strftime("%Y-%m")
)


# ============================================================
# MERGE
# ============================================================

weekly_cpi = weekly_cpi.merge(

    cpi_df[

        [
            "month_key",
            "headline_cpi",
            "core_cpi_trim"
        ]

    ],

    on="month_key",

    how="left"
)


# ============================================================
# FINAL CLEANUP
# ============================================================

weekly_cpi = weekly_cpi[

    [
        "week_start",
        "headline_cpi",
    ]
]

weekly_cpi["week_start"] = (
    weekly_cpi["week_start"]
    .dt.date
)


# ============================================================
# PREVIEW
# ============================================================

print("\nWEEKLY CPI DATA\n")

print(weekly_cpi.head(20))


# ============================================================
# COLUMN DESCRIPTIONS
# ============================================================

print("\nCOLUMN DESCRIPTIONS\n")

print(
    "week_start: Monday start date of the week"
)

print(
    "headline_cpi: Headline CPI inflation YoY (%)"
)

In [0]:
weekly_cpi.display()